In [3]:
from pathlib import Path
import pandas as pd

current_dir = Path.cwd()

for folder in [current_dir, *current_dir.parents]:
    train_path = folder / "data" / "raw" / "train.csv"
    if train_path.is_file():
        break
train = pd.read_csv(train_path)

for folder in [current_dir, *current_dir.parents]:
    train_demographics_path = folder / "data" / "raw" / "train_demographics.csv"
    if train_demographics_path.is_file():
        break
train_demographics = pd.read_csv(train_demographics_path)

print("Kích thước:", train.shape)
print("Số chuỗi:", train["sequence_id"].nunique())
print("Số người:", train["subject"].nunique())

display(train.head())
train.info()

Kích thước: (574945, 341)
Số chuỗi: 8151
Số người: 81


,row_id,sequence_type,sequence_id,sequence_counter,subject,orientation,behavior,phase,gesture,acc_x,...,tof_5_v54,tof_5_v55,tof_5_v56,tof_5_v57,tof_5_v58,tof_5_v59,tof_5_v60,tof_5_v61,tof_5_v62,tof_5_v63
0,SEQ_000007_000000,Target,SEQ_000007,0,SUBJ_059520,Seated Lean Non Dom - FACE DOWN,Relaxes and moves hand to target location,Transition,Cheek - pinch skin,6.683594,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0
1,SEQ_000007_000001,Target,SEQ_000007,1,SUBJ_059520,Seated Lean Non Dom - FACE DOWN,Relaxes and moves hand to target location,Transition,Cheek - pinch skin,6.949219,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0
2,SEQ_000007_000002,Target,SEQ_000007,2,SUBJ_059520,Seated Lean Non Dom - FACE DOWN,Relaxes and moves hand to target location,Transition,Cheek - pinch skin,5.722656,...,-1.0,-1.0,112.0,119.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0
3,SEQ_000007_000003,Target,SEQ_000007,3,SUBJ_059520,Seated Lean Non Dom - FACE DOWN,Relaxes and moves hand to target location,Transition,Cheek - pinch skin,6.601562,...,-1.0,-1.0,101.0,111.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0
4,SEQ_000007_000004,Target,SEQ_000007,4,SUBJ_059520,Seated Lean Non Dom - FACE DOWN,Relaxes and moves hand to target location,Transition,Cheek - pinch skin,5.566406,...,-1.0,-1.0,101.0,109.0,125.0,-1.0,-1.0,-1.0,-1.0,-1.0


<class 'pandas.DataFrame'>
RangeIndex: 574945 entries, 0 to 574944
Columns: 341 entries, row_id to tof_5_v63
dtypes: float64(332), int64(1), str(8)
memory usage: 1.5 GB


In [4]:
# Kiểm tra trùng lặp
cols = train.columns.drop("row_id")
print("Số hàng trùng khi bỏ row_id:", train.duplicated(subset=cols).sum())

# Kiểm tra mỗi chuỗi có đúng một người và một nhãn
check = train.groupby("sequence_id")[["subject", "gesture"]].nunique()
print(check.max())

Số hàng trùng khi bỏ row_id: 0
subject    1
gesture    1
dtype: int64


In [5]:
# Tạo bảng: mỗi hàng đại diện cho một chuỗi
sequences = (
    train.groupby("sequence_id", sort=False)
    .agg(
        subject=("subject", "first"),
        gesture=("gesture", "first"),
        length=("sequence_counter", "size")
    )
)

display(sequences.head())

,subject,gesture,length
sequence_id,,,
SEQ_000007,SUBJ_059520,Cheek - pinch skin,57
SEQ_000008,SUBJ_020948,Forehead - pull hairline,68
SEQ_000013,SUBJ_040282,Cheek - pinch skin,53
SEQ_000016,SUBJ_052342,Write name on leg,61
SEQ_000018,SUBJ_032165,Forehead - pull hairline,54


In [6]:
# Thống kê số lượng chuỗi theo nhãn
display(sequences["gesture"].value_counts())

gesture
Forehead - pull hairline                      640
Neck - scratch                                640
Neck - pinch skin                             640
Eyelash - pull hair                           640
Forehead - scratch                            640
Text on phone                                 640
Eyebrow - pull hair                           638
Above ear - pull hair                         638
Cheek - pinch skin                            637
Wave hello                                    478
Write name in air                             477
Pull air toward your face                     477
Write name on leg                             161
Feel around in tray and pull out an object    161
Pinch knee/leg skin                           161
Scratch knee/leg skin                         161
Drink from bottle/cup                         161
Glasses on/off                                161
Name: count, dtype: int64

In [7]:
# Thống kê độ dài chuỗi
display(sequences["length"].describe())

count    8151.000000
mean       70.536744
std        35.389879
min        29.000000
25%        51.000000
50%        59.000000
75%        78.000000
max       700.000000
Name: length, dtype: float64

In [8]:
# Thống kê số lượng chuỗi theo người
display(sequences.groupby("subject").size().describe())

count     81.000000
mean     100.629630
std        7.757971
min       51.000000
25%      102.000000
50%      102.000000
75%      102.000000
max      102.000000
dtype: float64

In [9]:
# Đếm số sequence vượt các ngưỡng độ dài
for threshold in [100, 150, 200, 300]:
    count = (sequences["length"] > threshold).sum()
    percentage = count / len(sequences) * 100

    print(f"Length > {threshold}: {count} ({percentage:.2f}%)")

# Xem 15 sequence dài nhất
display(sequences.sort_values("length", ascending=False).head(15))


Length > 100: 882 (10.82%)
Length > 150: 246 (3.02%)
Length > 200: 96 (1.18%)
Length > 300: 23 (0.28%)


,subject,gesture,length
sequence_id,,,
SEQ_015261,SUBJ_040733,Neck - scratch,700
SEQ_014205,SUBJ_017170,Neck - scratch,671
SEQ_028188,SUBJ_042794,Neck - scratch,647
SEQ_060593,SUBJ_040733,Neck - scratch,630
SEQ_016031,SUBJ_032761,Neck - scratch,567
SEQ_013984,SUBJ_023739,Neck - scratch,429
SEQ_065478,SUBJ_052342,Neck - scratch,426
SEQ_002216,SUBJ_021670,Neck - scratch,398
SEQ_008179,SUBJ_028998,Neck - scratch,393


In [10]:
# Thống kê độ dài theo từng gesture
length_by_gesture = sequences.groupby("gesture")["length"].agg(
    count="count",
    mean="mean",
    median="median",
    min="min",
    max="max",
    p90=lambda x: x.quantile(0.90),
    n_over_150=lambda x: (x > 150).sum()
)

# Tính tỷ lệ sequence dài hơn 150 của từng gesture
length_by_gesture["pct_over_150"] = (
    length_by_gesture["n_over_150"]
    / length_by_gesture["count"] * 100
)

# Sắp xếp theo độ dài trung vị giảm dần
length_by_gesture = length_by_gesture.sort_values(
    "median", ascending=False
)

display(length_by_gesture.round(2))

,count,mean,median,min,max,p90,n_over_150,pct_over_150
gesture,,,,,,,,
Feel around in tray and pull out an object,161,106.30,86.0,50,322,186.0,27,16.77
Text on phone,640,91.35,75.0,29,390,151.1,66,10.31
Drink from bottle/cup,161,81.32,69.0,34,176,124.0,5,3.11
Glasses on/off,161,84.11,68.0,43,293,148.0,16,9.94
Neck - scratch,640,88.47,62.0,41,700,146.1,61,9.53
Wave hello,478,71.87,60.0,34,230,110.0,12,2.51
Pull air toward your face,477,64.45,58.0,36,188,92.4,1,0.21
Scratch knee/leg skin,161,76.57,58.0,42,229,135.0,8,4.97
Forehead - pull hairline,640,63.75,57.0,35,190,91.1,3,0.47


### Nhận xét

- Dữ liệu gồm 8.151 chuỗi của 81 người. Không phát hiện hàng trùng khi bỏ cột `row_id`; mỗi chuỗi có đúng một người và một nhãn hành vi.
- Số chuỗi giữa các nhãn chưa cân bằng: nhãn nhiều nhất có 640 chuỗi, ít nhất có 161 chuỗi, chênh lệch gần 4 lần.
- Độ dài trung vị là 59 bước, trung bình là 70,54 bước; khoảng 50% số chuỗi có độ dài từ 51 đến 78 bước.
- Chỉ 3,02% số chuỗi dài hơn 150 bước, nhưng chuỗi dài nhất tới 700 bước. Một số chuỗi rất dài kéo độ dài trung bình lên; chưa thể coi chúng là dữ liệu lỗi.
- Độ dài khác nhau giữa các hành vi. Nhãn `Feel around in tray and pull out an object` có 16,77% chuỗi dài hơn 150 bước, trong khi `Pinch knee/leg skin` không có chuỗi nào vượt ngưỡng này. Vì vậy, cắt chuỗi ở 150 bước sẽ ảnh hưởng khác nhau đến từng nhãn.
